# Setup


In [ ]:
# Working directory should be the root directory of repository.
# setwd("./")
renv::load()
source("./results/utils.R")

suppressPackageStartupMessages({
  library(CAdir)
  library(APL)

  library(SingleCellExperiment)
  library(scater)
  library(scuttle)
  library(scran)
  library(batchelor)

  library(dplyr)

  library(patchwork)
})

options(repr.plot.width = 20, repr.plot.height = 15)

dir <- "./results/"
imgdir <- file.path(dir, "img/suppl_mat/batch/")
dir.create(imgdir, recursive = TRUE)

## Load data
<https://figshare.com/articles/dataset/Benchmarking_atlas-level_data_integration_in_single-cell_genomics_-_integration_task_datasets_Immune_and_pancreas_/12420968?file=24539828>

In [ ]:
data_dir <- "./data/real/raw/"

sce <- readRDS(file.path(data_dir, "human_pancreas_norm_complexBatch.rds"))


genevars <- modelGeneVar(sce, assay.type = "logcounts")
chosen <- getTopHVGs(genevars, n = 6000, var.threshold = NULL)
sce_sub <- sce[chosen, ]

# sce_sub <- fixedPCA(sce_sub, rank = 50, subset.row = chosen)
# sce_sub <- runUMAP(sce_sub, dimred = "PCA")

# CA 

In [ ]:
ca <- cacomp(
  obj = logcounts(sce_sub),
  princ_coords = 3,
  dims = 50,
  top = nrow(sce_sub),
  residuals = "pearson"
)

cell_types <- sce$celltype
cat("Number of cell types:", length(unique(cell_types)), "\n")

# CAdir


In [ ]:
set.seed(2358)
cabic <- dirclust_splitmerge(
  caobj = ca,
  k = 14,
  cutoff = 55,
  method = "random",
  apl_quant = 0.99,
  counts = NULL,
  min_cells = 20,
  reps = NULL,
  make_plots = TRUE,
  qcutoff = 0.8,
  convergence_thr = 0.001
)

Annotate biclustering:


In [ ]:
cabic <- annotate_biclustering(
  obj = cabic,
  universe = rownames(sce_sub),
  org = "hs"
)

sce_sub$cadir <- cabic@cell_clusters

cabic

In [ ]:
um1 <- plotUMAP(sce_sub, colour_by = "cadir") +
  guides(colour = guide_legend(title = "CAdir"))
um2 <- plotUMAP(sce_sub, colour_by = "celltype") +
  guides(colour = guide_legend(title = "Cell Type"))
um3 <- plotUMAP(sce_sub, colour_by = "tech") +
  guides(colour = guide_legend(title = "Technology"))

ari1 <- aricode::clustComp(sce_sub$cadir, sce_sub$celltype)
cat("ARI:", ari1$ARI, "\n")

um <- (um2 | um3) /
  (um1 + ggtitle(paste0("ARI: ", round(ari1$ARI, 2)))) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(face = "bold", size = 12))

um

ggsave(
  plot = um,
  filename = file.path(imgdir, "batch_umap.png"),
  width = 2400,
  height = 1600,
  units = "px"
)

# Cluster Association Plots


In [ ]:
pcl <- plot_clusters(
  cadir = cabic,
  caobj = ca,
  show_genes = TRUE,
  label_genes = TRUE,
  ntop = 5,
  title_prefix = "",
  axis = TRUE,
  text_size = 19,
  legend_pos = "right",
  return_list = TRUE,
  gsub_title = "_"
)
pcl <- wrap_plots(pcl, ncol = 3) +
  guide_area() +
  plot_layout(guides = "collect")

pcl

ggsave(
  plot = pcl,
  filename = file.path(imgdir, "cluster_apls.pdf"),
  device = cairo_pdf,
  width = 3400,
  height = 2000,
  units = "px"
)

ggsave(
  plot = pcl,
  filename = file.path(imgdir, "cluster_apls.png"),
  width = 3400,
  height = 2000,
  units = "px"
)

In [ ]:
cact <- cabic
cact@cell_clusters <- sce_sub$celltype
names(cact@cell_clusters) <- names(cabic@cell_clusters)

dirs <- list()
for (c in unique(sce_sub$celltype)) {
  grp <- which(sce_sub$celltype == c)
  dir <- colMeans(ca@prin_coords_cols[grp, ])
  dirs[[c]] <- dir
}
dirs <- do.call("rbind", dirs)
cact@directions <- dirs
cact@gene_clusters <- CAdir::assign_genes(ca, cact, qcutoff = 0.8)

capl1 <- cluster_apl(
  cadir = cact,
  caobj = ca,
  cluster = "beta",
  direction = cact@directions["beta", ],
  group = which(cact@cell_clusters == "beta"),
  highlight_cluster = TRUE,
  show_genes = TRUE,
  point_size = 1
) +
  ggtitle("beta - no batch correction") +
  theme(legend.position = "none")

# cluster_apl(cadir = cabic, caobj = ca, cluster = "Mast_cell")

In [ ]:
cact2 <- cabic
cact2@cell_clusters <- sce_sub$tech
names(cact2@cell_clusters) <- names(cabic@cell_clusters)

dirs <- list()
for (c in unique(sce_sub$tech)) {
  grp <- which(sce_sub$tech == c)
  dir <- colMeans(ca@prin_coords_cols[grp, ])
  dirs[[c]] <- dir
}
dirs <- do.call("rbind", dirs)
cact2@directions <- dirs
cact2@gene_clusters <- CAdir::assign_genes(ca, cact2, qcutoff = 0.8)

capl12 <- cluster_apl(
  cadir = cact2,
  caobj = ca,
  cluster = "smartseq2",
  direction = cact2@directions["smartseq2", ],
  group = which(cact2@cell_clusters == "smartseq2"),
  highlight_cluster = TRUE,
  show_genes = TRUE,
  point_size = 1
) +
  ggtitle("smartseq2 - no batch correction") +
  theme(legend.position = "none")

# cluster_apl(cadir = cabic, caobj = ca, cluster = "Mast_cell")

# Batch correction


In [ ]:
batch_corr <- quickCorrect(
  sce_sub,
  batch = sce_sub$tech,
  PARAM = FastMnnParam(BSPARAM = BiocSingular::RandomParam()),
  hvg.args = list(n = 5000)
)

sce_corr <- batch_corr$corrected

In [ ]:
stopifnot(identical(rownames(colData(sce_corr)), rownames(colData(sce_sub))))
sce_corr$celltype <- sce_sub$celltype

sce_corr <- runUMAP(sce_corr, dimred = "corrected")
um4 <- plotUMAP(sce_corr, colour = "batch") +
  guides(colour = guide_legend(title = "Batch"))
um5 <- plotUMAP(sce_corr, colour = "celltype") +
  guides(colour = guide_legend(title = "Cell Type"))

In [ ]:
reconstr <- as.matrix(assay(sce_corr, "reconstructed"))
reconstr <- reconstr + abs(min(reconstr))

ca_corr <- cacomp(
  obj = reconstr,
  princ_coords = 3,
  dims = 50,
  top = nrow(sce_corr),
  residuals = "pearson"
)

cell_types <- sce$celltype
cat("Number of cell types:", length(unique(cell_types)), "\n")

# CAdir


In [ ]:
set.seed(2358)
cabic_corr <- dirclust_splitmerge(
  caobj = ca_corr,
  k = 14,
  cutoff = 55,
  method = "random",
  apl_quant = 0.99,
  counts = NULL,
  min_cells = 20,
  reps = NULL,
  make_plots = TRUE,
  qcutoff = 0.8,
  convergence_thr = 0.001
)

In [ ]:
sce_corr$cadir <- cabic_corr@cell_clusters
ari2 <- aricode::clustComp(sce_corr$cadir, sce_corr$celltype)
cat("ARI:", ari2$ARI, "\n")

In [ ]:
cabic_corr <- annotate_biclustering(
  obj = cabic_corr,
  universe = rownames(sce_corr),
  org = "hs"
)

sce_corr$cadir <- cabic_corr@cell_clusters
cabic

In [ ]:
um6 <- plotUMAP(sce_corr, colour = "cadir") +
  guides(colour = guide_legend(title = "CAdir"))

um <- (um4 | um5) /
  (um6 + ggtitle(paste0("ARI: ", round(ari2$ARI, 2)))) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 12))
um

ggsave(
  plot = um,
  filename = file.path(imgdir, "batch_corr_umap.png"),
  width = 2400,
  height = 1600,
  units = "px"
)

In [ ]:
cact <- cabic_corr
cact@cell_clusters <- sce_corr$celltype
names(cact@cell_clusters) <- names(cabic_corr@cell_clusters)

dirs <- list()
for (c in unique(sce_corr$celltype)) {
  grp <- which(sce_corr$celltype == c)
  dir <- colMeans(ca_corr@prin_coords_cols[grp, ])
  dirs[[c]] <- dir
}
dirs <- do.call("rbind", dirs)
cact@directions <- dirs
cact@gene_clusters <- CAdir::assign_genes(ca_corr, cact, qcutoff = 0.8)

capl2 <- cluster_apl(
  cadir = cact,
  caobj = ca_corr,
  cluster = "beta",
  direction = cact@directions["beta", ],
  group = which(cact@cell_clusters == "beta"),
  highlight_cluster = TRUE,
  show_genes = TRUE,
  point_size = 1
) +
  ggtitle("beta - batch corrected") +
  theme(legend.position = "none")
# cluster_apl(cadir = cabic, caobj = ca, cluster = "Mast_cell")

In [ ]:
cact2 <- cabic_corr
cact2@cell_clusters <- as.factor(sce_corr$batch)
names(cact2@cell_clusters) <- names(cabic_corr@cell_clusters)

dirs <- list()
for (c in unique(sce_corr$batch)) {
  grp <- which(sce_corr$batch == c)
  dir <- colMeans(ca_corr@prin_coords_cols[grp, ])
  dirs[[c]] <- dir
}
dirs <- do.call("rbind", dirs)
cact2@directions <- dirs
cact2@gene_clusters <- CAdir::assign_genes(ca_corr, cact2, qcutoff = 0.8)

capl22 <- cluster_apl(
  cadir = cact2,
  caobj = ca_corr,
  cluster = "smartseq2",
  direction = cact2@directions["smartseq2", ],
  group = which(cact2@cell_clusters == "smartseq2"),
  highlight_cluster = TRUE,
  show_genes = TRUE,
  point_size = 1
) +
  ggtitle("smartseq2 - batch corrected") +
  theme(legend.position = "none")
# cluster_apl(cadir = cabic, caobj = ca, cluster = "Mast_cell")

In [ ]:
capl3 <- cluster_apl(cadir = cabic, caobj = ca, cluster = "Beta_cell(β_cell)") +
  ggtitle("Beta_cell(β_cell) - no batch correction")
capl4 <- cluster_apl(
  cadir = cabic_corr,
  caobj = ca_corr,
  cluster = "Beta_cell(β_cell)"
) +
  ggtitle("Beta_cell(β_cell) - batch correction")

# capl3 <- cluster_apl(cadir = cabic, caobj = ca, cluster = "Acinar_cell")
# capl4 <- cluster_apl(cadir = cabic_corr, caobj = ca_corr, cluster = "Acinar_cell")
capl3 + capl4

In [ ]:
um_batch <- (um2 | um3) /
  (um1 + ggtitle(paste0("ARI: ", round(ari1$ARI, 2))) + (capl1 / capl12)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(face = "bold", size = 12))

um_no_batch <- (um5 | um4) /
  (um6 + ggtitle(paste0("ARI: ", round(ari2$ARI, 2))) + (capl2 / capl22)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(face = "bold", size = 12))

ggsave(
  plot = um_batch,
  filename = file.path(imgdir, "fig1_batch_umap.png"),
  width = 2600,
  height = 2000,
  units = "px"
)
ggsave(
  plot = um_no_batch,
  filename = file.path(imgdir, "fig2_no_batch_umap.png"),
  width = 2600,
  height = 2000,
  units = "px"
)

In [ ]:
all <- (((um2 | um3) /
  (um1 + ggtitle(paste0("ARI: ", round(ari1$ARI, 2))) + capl1)) |
  ((um4 | um5) /
    (um6 + ggtitle(paste0("ARI: ", round(ari2$ARI, 2))) + capl2))) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(face = "bold", size = 12))

ggsave(
  plot = all,
  filename = file.path(imgdir, "all_fig_umap.png"),
  width = 4800,
  height = 1600,
  units = "px"
)

In [ ]:
cabic <- rank_genes(cabic, ca)
top <- cabic@gene_ranks$"Beta_cell(β_cell)"
nrow(top[top$Score >= 0, ])

cabic_corr <- rank_genes(cabic_corr, ca_corr)
top_corr <- cabic_corr@gene_ranks$"Beta_cell(β_cell)"
nrow(top_corr[top_corr$Score >= 0, ])

In [ ]:
sessionInfo()